
# Pronóstico climático mensual 2025–2027

Este notebook toma como base `clima_mes_2005-2025.csv` y genera:

- entrenamiento de modelos por **municipio-variable**
- comparación entre modelos
- selección automática del mejor modelo
- pronóstico mensual hasta 2027
- archivo final `clima_mes_2005-2027.csv`

La lógica del notebook es pronosticar variables climáticas mensuales base y no las features derivadas del índice.


## 1. Librerías

In [ ]:

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    STATSMODELS_OK = True
except Exception:
    STATSMODELS_OK = False

print("Statsmodels disponible:", STATSMODELS_OK)


## 2. Rutas, parámetros y carga de datos

In [ ]:

PROJECT_ROOT = Path().resolve().parents[1]
path = PROJECT_ROOT / "data" / "processed"

archivo_entrada = path / "clima_mes_2005-2025.csv"

df = pd.read_csv(archivo_entrada)

df.head()


In [ ]:

# Parámetros principales

COL_MUNICIPIO = "municipio"
COL_ANIO = "anio"
COL_MES = "mes"

# Si se deja en None, el pronóstico empieza en el mes siguiente al último dato observado.
# Si quieres forzar pronóstico desde enero de 2025, usa: "2025-01-01"
FECHA_INICIO_PRONOSTICO = None

FECHA_FIN_PRONOSTICO = "2027-12-01"

TEST_SIZE = 0.10
RANDOM_STATE = 42

# Variables climáticas base esperadas.
# El notebook usará únicamente las que existan en el archivo.
variables_base_esperadas = [
    "precip_mm",
    "precip_mean_mm",
    "precip_max_mm",
    "temp_mean",
    "temp_min",
    "temp_max",
    "et_real_mm",
    "et_potencial_mm",
    "ndvi_mean",
    "ndvi_min",
    "ndvi_max"
]


## 3. Validación y preparación temporal

In [ ]:

# Validaciones mínimas
cols_base = [COL_MUNICIPIO, COL_ANIO, COL_MES]
faltantes = [c for c in cols_base if c not in df.columns]

if faltantes:
    raise ValueError(f"Faltan columnas base en el archivo mensual: {faltantes}")

# Variables climáticas disponibles
variables_clima = [v for v in variables_base_esperadas if v in df.columns]

if len(variables_clima) == 0:
    raise ValueError("No se encontró ninguna variable climática base esperada.")

print("Variables climáticas a pronosticar:")
print(variables_clima)

# Crear fecha mensual
df[COL_ANIO] = df[COL_ANIO].astype(int)
df[COL_MES] = df[COL_MES].astype(int)

df["date"] = pd.to_datetime(
    df[COL_ANIO].astype(str) + "-" + df[COL_MES].astype(str).str.zfill(2) + "-01"
)

# Ordenar
df = df.sort_values([COL_MUNICIPIO, "date"]).reset_index(drop=True)

print("Rango observado:")
print(df["date"].min(), "→", df["date"].max())
print("Municipios:", df[COL_MUNICIPIO].nunique())
print("Filas:", len(df))


In [ ]:

# Convertir variables climáticas a numéricas
for v in variables_clima:
    df[v] = pd.to_numeric(df[v], errors="coerce")

# Revisar faltantes
faltantes_vars = df[variables_clima].isna().mean().sort_values(ascending=False)
faltantes_vars


## 4. Funciones auxiliares de pronóstico

In [ ]:

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def split_train_test(serie, test_size=0.10):
    n = len(serie)
    n_test = max(1, int(np.ceil(n * test_size)))
    train = serie.iloc[:-n_test]
    test = serie.iloc[-n_test:]
    return train, test


def forecast_seasonal_naive(train, fechas_objetivo):
    """
    Pronóstico por climatología mensual:
    para cada mes futuro usa el promedio histórico de ese mes.
    """
    train_df = train.to_frame("y")
    train_df["mes"] = train_df.index.month

    clim = train_df.groupby("mes")["y"].mean()
    fallback = train.mean()

    preds = []
    for fecha in fechas_objetivo:
        preds.append(clim.get(fecha.month, fallback))

    return pd.Series(preds, index=fechas_objetivo)


def forecast_sarima(train, fechas_objetivo):
    """
    SARIMA con una grilla pequeña para evitar tiempos excesivos.
    """
    if not STATSMODELS_OK:
        raise RuntimeError("statsmodels no está disponible.")

    best_aic = np.inf
    best_model = None

    orders = [
        (1, 0, 0),
        (0, 1, 1),
        (1, 1, 0),
        (1, 1, 1)
    ]

    seasonal_orders = [
        (1, 0, 0, 12),
        (0, 1, 1, 12),
        (1, 1, 0, 12)
    ]

    for order in orders:
        for seasonal_order in seasonal_orders:
            try:
                model = SARIMAX(
                    train,
                    order=order,
                    seasonal_order=seasonal_order,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
                fit = model.fit(disp=False)

                if fit.aic < best_aic:
                    best_aic = fit.aic
                    best_model = fit
            except Exception:
                continue

    if best_model is None:
        raise RuntimeError("No fue posible ajustar SARIMA.")

    pred = best_model.forecast(steps=len(fechas_objetivo))
    pred.index = fechas_objetivo

    return pred


def crear_dataset_rf(serie, lags=(1, 2, 3, 6, 12)):
    df_lag = pd.DataFrame({"y": serie})

    for lag in lags:
        df_lag[f"lag_{lag}"] = df_lag["y"].shift(lag)

    df_lag["mes"] = df_lag.index.month
    df_lag["mes_sin"] = np.sin(2 * np.pi * df_lag["mes"] / 12)
    df_lag["mes_cos"] = np.cos(2 * np.pi * df_lag["mes"] / 12)

    df_lag = df_lag.dropna()

    X = df_lag.drop(columns=["y"])
    y = df_lag["y"]

    return X, y


def forecast_rf_recursive(train, fechas_objetivo, random_state=42, lags=(1, 2, 3, 6, 12)):
    X_train, y_train = crear_dataset_rf(train, lags=lags)

    if len(X_train) < 24:
        raise RuntimeError("Serie insuficiente para Random Forest con rezagos.")

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=3,
        random_state=random_state,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    historial = train.copy()
    preds = []

    for fecha in fechas_objetivo:
        row = {}
        for lag in lags:
            row[f"lag_{lag}"] = historial.iloc[-lag]

        mes = fecha.month
        row["mes"] = mes
        row["mes_sin"] = np.sin(2 * np.pi * mes / 12)
        row["mes_cos"] = np.cos(2 * np.pi * mes / 12)

        X_fut = pd.DataFrame([row])
        pred = model.predict(X_fut)[0]

        preds.append(pred)
        historial.loc[fecha] = pred

    return pd.Series(preds, index=fechas_objetivo)


def ajustar_no_negativos(pred, variable):
    """
    Variables que por definición no deben ser negativas.
    """
    vars_no_negativas = [
        "precip_mm", "precip_mean_mm", "precip_max_mm",
        "et_real_mm", "et_potencial_mm",
        "ndvi_mean", "ndvi_min", "ndvi_max"
    ]

    if variable in vars_no_negativas:
        pred = pred.clip(lower=0)

    # NDVI usualmente está entre 0 y 1
    if variable.startswith("ndvi"):
        pred = pred.clip(lower=0, upper=1)

    return pred


## 5. Función para seleccionar mejor modelo por municipio-variable

In [ ]:

def evaluar_modelos_para_serie(serie, variable, test_size=0.10, random_state=42):
    """
    Entrena y evalúa modelos sobre la parte de prueba.
    Retorna métricas y el mejor modelo por RMSE.
    """
    serie = serie.dropna().astype(float)

    if len(serie) < 36:
        raise RuntimeError("Serie insuficiente: menos de 36 meses.")

    train, test = split_train_test(serie, test_size=test_size)
    fechas_test = test.index

    resultados = []
    predicciones_test = {}

    # Modelo 1: baseline estacional
    try:
        pred = forecast_seasonal_naive(train, fechas_test)
        pred = ajustar_no_negativos(pred, variable)
        resultados.append({
            "modelo": "seasonal_naive",
            "mae": mean_absolute_error(test, pred),
            "rmse": rmse(test, pred),
            "sesgo": (pred - test).mean()
        })
        predicciones_test["seasonal_naive"] = pred
    except Exception as e:
        pass

    # Modelo 2: SARIMA
    try:
        pred = forecast_sarima(train, fechas_test)
        pred = ajustar_no_negativos(pred, variable)
        resultados.append({
            "modelo": "sarima",
            "mae": mean_absolute_error(test, pred),
            "rmse": rmse(test, pred),
            "sesgo": (pred - test).mean()
        })
        predicciones_test["sarima"] = pred
    except Exception as e:
        pass

    # Modelo 3: Random Forest con rezagos
    try:
        pred = forecast_rf_recursive(train, fechas_test, random_state=random_state)
        pred = ajustar_no_negativos(pred, variable)
        resultados.append({
            "modelo": "random_forest_lags",
            "mae": mean_absolute_error(test, pred),
            "rmse": rmse(test, pred),
            "sesgo": (pred - test).mean()
        })
        predicciones_test["random_forest_lags"] = pred
    except Exception as e:
        pass

    if len(resultados) == 0:
        raise RuntimeError("Ningún modelo pudo ajustarse.")

    resultados = pd.DataFrame(resultados).sort_values("rmse")
    mejor_modelo = resultados.iloc[0]["modelo"]

    return resultados, mejor_modelo, predicciones_test


def pronosticar_con_modelo(serie, variable, modelo, fechas_futuras, random_state=42):
    """
    Reentrena el modelo seleccionado con toda la serie observada
    y pronostica las fechas futuras.
    """
    serie = serie.dropna().astype(float)

    if modelo == "seasonal_naive":
        pred = forecast_seasonal_naive(serie, fechas_futuras)

    elif modelo == "sarima":
        pred = forecast_sarima(serie, fechas_futuras)

    elif modelo == "random_forest_lags":
        pred = forecast_rf_recursive(serie, fechas_futuras, random_state=random_state)

    else:
        raise ValueError(f"Modelo no reconocido: {modelo}")

    pred = ajustar_no_negativos(pred, variable)

    return pred


## 6. Definir horizonte futuro

In [ ]:

fecha_max_obs = df["date"].max()

if FECHA_INICIO_PRONOSTICO is None:
    fecha_inicio = fecha_max_obs + pd.offsets.MonthBegin(1)
else:
    fecha_inicio = pd.to_datetime(FECHA_INICIO_PRONOSTICO)

fecha_fin = pd.to_datetime(FECHA_FIN_PRONOSTICO)

fechas_futuras = pd.date_range(
    start=fecha_inicio,
    end=fecha_fin,
    freq="MS"
)

print("Fecha máxima observada:", fecha_max_obs)
print("Inicio pronóstico:", fecha_inicio)
print("Fin pronóstico:", fecha_fin)
print("Meses a pronosticar:", len(fechas_futuras))


## 7. Entrenar, seleccionar y pronosticar por municipio-variable

In [ ]:

metricas = []
pronosticos = []

municipios = sorted(df[COL_MUNICIPIO].dropna().unique())

for municipio in municipios:

    df_m = df[df[COL_MUNICIPIO] == municipio].copy()
    df_m = df_m.sort_values("date")

    for variable in variables_clima:

        serie = (
            df_m
            .set_index("date")[variable]
            .sort_index()
        )

        # Asegurar frecuencia mensual
        serie = serie.asfreq("MS")

        try:
            resultados, mejor_modelo, _ = evaluar_modelos_para_serie(
                serie=serie,
                variable=variable,
                test_size=TEST_SIZE,
                random_state=RANDOM_STATE
            )

            resultados["municipio"] = municipio
            resultados["variable"] = variable
            resultados["modelo_seleccionado"] = mejor_modelo

            metricas.append(resultados)

            pred_fut = pronosticar_con_modelo(
                serie=serie,
                variable=variable,
                modelo=mejor_modelo,
                fechas_futuras=fechas_futuras,
                random_state=RANDOM_STATE
            )

            df_pred = pd.DataFrame({
                COL_MUNICIPIO: municipio,
                "date": pred_fut.index,
                "variable": variable,
                "valor": pred_fut.values,
                "modelo_seleccionado": mejor_modelo
            })

            pronosticos.append(df_pred)

        except Exception as e:
            print(f"Error en {municipio} - {variable}: {e}")

metricas_modelos = pd.concat(metricas, ignore_index=True)
pronosticos_long = pd.concat(pronosticos, ignore_index=True)

metricas_modelos.head()


In [ ]:

# Modelos seleccionados por frecuencia
(
    metricas_modelos
    .drop_duplicates(["municipio", "variable"])
    ["modelo_seleccionado"]
    .value_counts()
)


In [ ]:

# Métricas promedio por modelo
metricas_modelos.groupby("modelo")[["mae", "rmse", "sesgo"]].mean().sort_values("rmse")


## 8. Reorganizar pronósticos a formato ancho

In [ ]:

pronosticos_wide = (
    pronosticos_long
    .pivot_table(
        index=[COL_MUNICIPIO, "date"],
        columns="variable",
        values="valor"
    )
    .reset_index()
)

pronosticos_wide[COL_ANIO] = pronosticos_wide["date"].dt.year
pronosticos_wide[COL_MES] = pronosticos_wide["date"].dt.month
pronosticos_wide["tipo_dato"] = "pronosticado"

# Orden de columnas
cols_id = [COL_MUNICIPIO, "date", COL_ANIO, COL_MES, "tipo_dato"]
cols_clima = [c for c in variables_clima if c in pronosticos_wide.columns]

pronosticos_wide = pronosticos_wide[cols_id + cols_clima]

pronosticos_wide.head()


## 9. Preparar datos observados y unir observado + pronosticado

In [ ]:

df_obs = df.copy()
df_obs["tipo_dato"] = "observado"

# Mantener columnas compatibles
cols_obs = [COL_MUNICIPIO, "date", COL_ANIO, COL_MES, "tipo_dato"] + variables_clima
df_obs = df_obs[cols_obs]

clima_mes_2005_2027 = pd.concat(
    [df_obs, pronosticos_wide],
    ignore_index=True
).sort_values([COL_MUNICIPIO, "date"]).reset_index(drop=True)

clima_mes_2005_2027.head()


In [ ]:

print("Rango final:")
print(clima_mes_2005_2027["date"].min(), "→", clima_mes_2005_2027["date"].max())

print("Filas por tipo de dato:")
print(clima_mes_2005_2027["tipo_dato"].value_counts())

print("Municipios:", clima_mes_2005_2027[COL_MUNICIPIO].nunique())


## 10. Validación visual rápida

In [ ]:

# Cambia estos valores si quieres inspeccionar otra serie
municipio_ejemplo = clima_mes_2005_2027[COL_MUNICIPIO].dropna().unique()[0]
variable_ejemplo = variables_clima[0]

df_plot = clima_mes_2005_2027[
    clima_mes_2005_2027[COL_MUNICIPIO] == municipio_ejemplo
].copy()

plt.figure(figsize=(10, 4))
plt.plot(df_plot["date"], df_plot[variable_ejemplo])
plt.axvline(fecha_inicio, linestyle="--")
plt.title(f"{municipio_ejemplo} - {variable_ejemplo}")
plt.xlabel("Fecha")
plt.ylabel(variable_ejemplo)
plt.grid()
plt.show()


## 11. Guardar archivos

In [ ]:

salida_clima_mes = path / "clima_mes_2005-2027.csv"
salida_pronostico = path / "clima_mes_pronostico_2025-2027.csv"
salida_metricas = path / "metricas_pronostico_climatico.csv"
salida_modelos = path / "modelos_seleccionados_pronostico.csv"

clima_mes_2005_2027.to_csv(salida_clima_mes, index=False)
pronosticos_wide.to_csv(salida_pronostico, index=False)
metricas_modelos.to_csv(salida_metricas, index=False)

modelos_seleccionados = (
    metricas_modelos
    .sort_values(["municipio", "variable", "rmse"])
    .groupby(["municipio", "variable"])
    .first()
    .reset_index()
)

modelos_seleccionados.to_csv(salida_modelos, index=False)

print("Archivos guardados:")
print(salida_clima_mes)
print(salida_pronostico)
print(salida_metricas)
print(salida_modelos)



## 12. Siguiente paso

Con `clima_mes_2005-2027.csv` se debe ejecutar el mismo pipeline de features intra-anuales para construir:

- `features_intra_anuales_2005-2027.csv`
- `features_intra_anuales_2025-2027.csv`

Estas features son las que alimentan el índice de riesgo y la prima.
